# Bitcoin Dataset

In [15]:
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import urllib.request
import gzip
import shutil
import os
import warnings

warnings.filterwarnings('ignore')

In [16]:
def download_bitcoin_alpha():
    url = "https://snap.stanford.edu/data/soc-sign-bitcoinalpha.csv.gz"
    csv_file = "soc-sign-bitcoinalpha.csv"
    if os.path.exists(csv_file): return csv_file
    
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(url, "temp.gz")
    
    with gzip.open("temp.gz", 'rb') as f_in:
        with open(csv_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    os.remove("temp.gz")
    return csv_file

## Build signed hypergraph

In [17]:
def bitcoin_hyper(file_path):
    df = pd.read_csv(file_path, names=['src', 'dst', 'rating', 'time']) # src:give review; dst: being reviewed; rating:pos/neg
    nodes = np.unique(np.concatenate([df['src'], df['dst']]))
    node_map = {node: i for i, node in enumerate(nodes)}
    
    hyperedges, signs = [], []
    for src, group in df.groupby('src'):
        u = node_map[src]
        pos = [node_map[d] for d in group[group['rating'] > 0]['dst']]
        if pos:
            hyperedges.append(list(set([u] + pos)))
            signs.append(1.0)
        neg = [node_map[d] for d in group[group['rating'] < 0]['dst']]
        if neg:
            hyperedges.append(list(set([u] + neg)))
            signs.append(-1.0)
            
    rows, cols = [], [] # incidence matrix
    for j, edge in enumerate(hyperedges):
        for i in edge:
            rows.append(i); cols.append(j)
    H = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(nodes), len(hyperedges)))
    return H, np.array(signs), nodes

## ASHD

In [18]:
class ASHD:
    def __init__(self, n_clusters=15, embedding_dim=64, n_iter=30, alpha=0.1): # alpha: weight update
        self.n_clusters = n_clusters
        self.embedding_dim = embedding_dim
        self.n_iter = n_iter
        self.alpha = alpha
        self.eps = 1e-10 # make sure no division by zero

    def fit_predict(self, H, initial_signs, seed=42):
        n_nodes, n_edges = H.shape
        np.random.seed(seed)
        
        # X
        X = np.random.normal(0, 0.01, (n_nodes, self.embedding_dim)) # start with random X
        X = X / (np.linalg.norm(X, axis=1, keepdims=True) + self.eps) # do L2 normalization to X
        
        # weight
        W = initial_signs.astype(float).copy()
        # hyperedge degree
        d_e = np.array(H.sum(axis=0)).flatten() + self.eps
        De_inv = sparse.diags(1.0 / d_e)
        
        for t in range(self.n_iter):
            # diffusion
            W_abs = np.abs(W)
            # vertex degree
            d_v = np.array(H.dot(W_abs)).flatten() + self.eps
            Dv_inv_sqrt = sparse.diags(1.0 / np.sqrt(d_v))
            
            W_diag = sparse.diags(W)
            # Dv^-1/2 @ H @ W @ De^-1 @ H.T @ Dv^-1/2 @ X
            temp = Dv_inv_sqrt.dot(X)
            temp = H.T.dot(temp)
            temp = De_inv.dot(temp)
            temp = W_diag.dot(temp)
            temp = H.dot(temp)
            X_new = Dv_inv_sqrt.dot(temp)
            
            # Update X
            X_new = np.nan_to_num(X_new)
            X = X_new / (np.linalg.norm(X_new, axis=1, keepdims=True) + self.eps)
            
            # Update weight
            if t > 0 and t % 3 == 0:
                for j in range(n_edges):
                    idx = H.getcol(j).indices
                    if len(idx) > 1:
                        sim = cosine_similarity(X[idx])
                        avg_sim = (np.sum(sim) - len(idx)) / (len(idx)*(len(idx)-1) + self.eps)
                        W[j] = np.clip(W[j] + self.alpha * avg_sim, -1.0, 1.0)
        
        labels = KMeans(n_clusters=self.n_clusters, n_init=10, random_state=seed).fit_predict(X)
        return labels, X

## Baseline & Evaluate

In [ ]:
class ExperimentEvaluator:
    def __init__(self, n_clusters=15, n_runs=10):
        self.n_clusters = n_clusters
        self.n_runs = n_runs
        self.eps = 1e-10

    # Plain KMeans
    def run_plain_kmeans(self, H, seed):
        return KMeans(n_clusters=self.n_clusters, n_init=1, random_state=seed).fit_predict(H)

    # Spectral Clustering
    def run_spectral(self, H, seed):
        d_e = np.array(H.sum(axis=0)).flatten() + self.eps
        d_v = np.array(H.sum(axis=1)).flatten() + self.eps
        
        Dv_inv_sqrt = sparse.diags(1.0 / np.sqrt(d_v))
        De_inv = sparse.diags(1.0 / d_e)
        P = Dv_inv_sqrt @ H @ De_inv @ H.T @ Dv_inv_sqrt
        vals, vecs = sparse.linalg.eigsh(P, k=self.n_clusters, which='LM')
        return KMeans(n_clusters=self.n_clusters, n_init=1, random_state=seed).fit_predict(vecs)

    # HyperGCN (Clique Expansion)
    def run_hypergcn_clique(self, H, seed):
        # Clique Expansion: A = H @ De^-1 @ H.T
        d_e = np.array(H.sum(axis=0)).flatten() + self.eps
        De_inv = sparse.diags(1.0 / d_e)
        A = H @ De_inv @ H.T
        A.setdiag(0)
        
        d_v = np.array(A.sum(axis=1)).flatten() + self.eps
        D_inv_sqrt = sparse.diags(1.0 / np.sqrt(d_v))
        L_norm = D_inv_sqrt @ A @ D_inv_sqrt
        
        vals, vecs = sparse.linalg.eigsh(L_norm, k=self.n_clusters, which='LM')
        return KMeans(n_clusters=self.n_clusters, n_init=1, random_state=seed).fit_predict(vecs)
    
    
    # evaluate
    def compute_metrics(self, H, labels, embeddings):
        # 1. Silhouette Score (Cosine)
        sil = silhouette_score(embeddings, labels, metric='cosine')
        
        # Bio-Consistency
        bc_scores = []
        for c in range(self.n_clusters):
            idx = np.where(labels == c)[0]
            if len(idx) > 1:
                sim = cosine_similarity(embeddings[idx])
                avg_sim = (np.sum(sim) - len(idx)) / (len(idx)*(len(idx)-1) + self.eps)
                bc_scores.append(avg_sim)
        bc = np.mean(bc_scores) if bc_scores else 0
        
        # Modularity 
        m = H.shape[1]
        vol_V = H.sum()
        mod = 0
        for c in range(self.n_clusters):
            idx = set(np.where(labels == c)[0])
            e_c = 0
            for j in range(m):
                edge_nodes = set(H.getcol(j).indices)
                if edge_nodes and edge_nodes.issubset(idx):
                    e_c += 1
            vol_c = H[list(idx), :].sum()
            mod += (e_c / m) - (vol_c / vol_V)**2
            
        return sil, bc, mod


In [ ]:
def run_benchmark(H, signs, n_clusters=15):
    evaluator = ExperimentEvaluator(n_clusters=n_clusters, n_runs=10)
    ashd_model = ASHD(n_clusters=n_clusters)
    
    methods = {
        "Plain KMeans": evaluator.run_plain_kmeans,
        "Spectral": evaluator.run_spectral,
        "HyperGCN (Clique)": evaluator.run_hypergcn_clique,
        "ASHD (Ours)": None
    }
    
    final_results = defaultdict(list)

    for name, func in methods.items():
        for seed in range(10):
            if name == "ASHD (Ours)":
                labels, emb = ashd_model.fit_predict(H, signs, seed=seed)
            else:
                labels = func(H, seed)
                emb = H.toarray() if sparse.issparse(H) else H
            
            metrics = evaluator.compute_metrics(H, labels, emb)
            final_results[name].append(metrics)
            
    print(f"{'Method':<20} | {'Silhouette':<15} | {'Bio-Consist':<15} | {'Modularity'}")
    
    for name, scores in final_results.items():
        scores = np.array(scores)
        means = np.mean(scores, axis=0)
        stds = np.std(scores, axis=0)
        print(f"{name:<20} | {means[0]:.4f}±{stds[0]:.4f} | {means[1]:.4f}±{stds[1]:.4f} | {means[2]:.4f}±{stds[2]:.4f}")


In [21]:
if __name__ == "__main__":
    path = download_bitcoin_alpha()
    H, signs, node_ids = bitcoin_hyper(path)
    run_benchmark(H, signs, n_clusters=15)

Method               | Silhouette      | Bio-Consist     | Modularity
Plain KMeans         | -0.0407±0.0184 | 0.1764±0.0267 | 0.6786±0.0480
Spectral             | 0.0106±0.0053 | 0.6353±0.0116 | 0.1653±0.2231
HyperGCN (Clique)    | 0.0099±0.0047 | 0.4730±0.0047 | 0.1123±0.1927
ASHD (Ours)          | 0.8417±0.1093 | 0.7650±0.0260 | 0.0966±0.0501
